# Real-ESRGAN 视频增强（Kaggle T4 × 2）

这个 Notebook 使用每张 GPU 一个常驻模型进程，对重叠图块进行双卡批量推理。先运行任意起点的 10 秒测试；确认画质、显存和音画同步后，把 `TEST_SECONDS` 改为 `0` 处理到视频末尾。请在 Kaggle Notebook 设置中启用 **GPU T4 x2** 和网络访问。

In [ ]:
# Kaggle/IPython 没有内置 %git；这里注册一个安全的 git 行魔法，从而可按要求使用 %git clone。
import shlex
import subprocess
from IPython.core.magic import register_line_magic

@register_line_magic
def git(line):
    return subprocess.run(["git", *shlex.split(line)], check=True)

In [ ]:
%git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN

In [ ]:
%cd /kaggle/working/Real-ESRGAN
%pip install -q -r requirements-video-kaggle.txt
%pip install -q -e . --no-deps

In [ ]:
# 环境与关键依赖检查
import cv2
import torch
import torchvision
from pathlib import Path

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("opencv:", cv2.__version__)
print("CUDA devices:", torch.cuda.device_count())
for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    print(gpu_id, props.name, f"{props.total_memory / 1024**3:.1f} GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "请在 Kaggle 设置中选择 GPU T4 x2"
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["ffprobe", "-version"], check=True, stdout=subprocess.DEVNULL)
encoders = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], check=True, capture_output=True, text=True).stdout
assert "libx264" in encoders, "当前 ffmpeg 缺少 libx264 编码器"
print("ffmpeg/ffprobe/libx264: OK")

## 参数

推荐先使用 `realesr-animevideov3 + 2× + tile 256 + overlap 32 + 每卡 batch 4 + FP16`。`INPUT_WIDTH=0`、`INPUT_HEIGHT=0` 保持原始输入尺寸；只设置其中一个会保持宽高比。修改 `INPUT_VIDEO` 为 Kaggle Dataset 中的真实路径。

In [ ]:
INPUT_VIDEO = "/kaggle/input/YOUR_DATASET/YOUR_VIDEO.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan_test.mp4"
MODEL = "realesr-animevideov3"
MODEL_PATH = ""                 # 留空则自动下载官方权重
DENOISE_STRENGTH = 1.0             # 仅 general-x4v3 使用
SCALE = 2.0
INPUT_WIDTH = 0
INPUT_HEIGHT = 0
TILE_SIZE = 256
OVERLAP = 32
BATCH_SIZE = 4                     # 每张 GPU 的图块批量
GPU_IDS = "0,1"
VIDEO_CODEC = "libx264"
CRF = 18                           # 越低质量越高、文件越大
PRESET = "medium"
AUDIO_CODEC = "aac"             # 精确裁剪同步；完整视频可尝试 copy
AUDIO_BITRATE = "192k"
START_TIME = 0.0                   # 可改成任意测试起点（秒）
TEST_SECONDS = 10.0                # 改为 0 处理到末尾
PROGRESS_INTERVAL = 60.0

assert Path(INPUT_VIDEO).is_file(), f"输入不存在，请修改 INPUT_VIDEO: {INPUT_VIDEO}"

In [ ]:
# 所有推理、双卡、测试区间、进度及编码参数均显式传给 Python CLI。
!python inference_realesrgan_video_kaggle.py \
  --input "{INPUT_VIDEO}" \
  --output "{OUTPUT_VIDEO}" \
  --model "{MODEL}" \
  --model-path "{MODEL_PATH}" \
  --denoise-strength {DENOISE_STRENGTH} \
  --scale {SCALE} \
  --fp16 \
  --input-width {INPUT_WIDTH} \
  --input-height {INPUT_HEIGHT} \
  --tile-size {TILE_SIZE} \
  --overlap {OVERLAP} \
  --batch-size {BATCH_SIZE} \
  --gpu-ids "{GPU_IDS}" \
  --video-codec "{VIDEO_CODEC}" \
  --crf {CRF} \
  --preset "{PRESET}" \
  --audio-codec "{AUDIO_CODEC}" \
  --audio-bitrate "{AUDIO_BITRATE}" \
  --start-time {START_TIME} \
  --test-seconds {TEST_SECONDS} \
  --progress-interval {PROGRESS_INTERVAL} \
  --ffmpeg-bin ffmpeg \
  --ffprobe-bin ffprobe

In [ ]:
# 输出流、时长和音频检查
assert Path(OUTPUT_VIDEO).is_file(), "输出视频未生成，请查看上一个单元格日志"
subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=index,codec_type,codec_name,width,height,avg_frame_rate", "-of", "json", OUTPUT_VIDEO], check=True)
from IPython.display import Video, display
display(Video(OUTPUT_VIDEO, embed=False))

## OOM 与利用率排查顺序

1. 先把 `BATCH_SIZE` 从 `4` 降到 `2`，再降到 `1`；模型仍只在每卡加载一次。
2. 再把 `TILE_SIZE` 从 `256` 降到 `192` 或 `128`，同时把 `OVERLAP` 调为 `24` 或 `16`。
3. 仍然 OOM 时限制输入，例如 `INPUT_WIDTH=1280, INPUT_HEIGHT=0`；最后才把倍率从 `2` 降低。
4. 不要因 OOM 关闭 FP16；关闭 FP16 通常会增加显存。只有发现 FP16 数值异常时才使用 `--no-fp16`。
5. GPU 利用率锯齿通常来自主进程的视频解码、CPU 融合或 x264 编码。可用 `PRESET=fast`/`veryfast` 降低编码瓶颈；增大 batch 前先观察显存余量。
6. `AUDIO_CODEC=aac` 会按精确时间重建音频时间戳，最适合任意起点测试。完整视频需要尽量避免重编码时可试 `copy`；若容器不兼容脚本会回退到 AAC。